# 🛡️ SmartOmniSentinel — Cloud Training Notebook (RWF-2000)
### Fast GPU Training & ONNX/PyTorch Export for Research Paper & Deployment

This notebook trains both core ML models for **SmartOmniSentinel** on free Google Colab GPU:
1. **`temporal_v1.pt`**: Pose-based Temporal CNN-GRU Classifier (Used for real-time edge CCTV inference).
2. **`violence_model.onnx`**: ResNet3D-18 (R3D-18) Video Classifier.

> **Step 0**: In the Colab top menu, go to **Runtime** > **Change runtime type** > Select **T4 GPU** (or any GPU) > Click **Save**.

In [ ]:
# Step 1: Install Dependencies
!pip install -q ultralytics onnx onnxruntime tqdm opencv-python

## Step 2A: Kaggle Authentication

Before running the download cell:
1. Go to https://www.kaggle.com/settings
2. Scroll to API section -> click Create New Token
3. A kaggle.json file downloads to your laptop
4. Run the cell below, click Choose Files, select kaggle.json


In [ ]:
from google.colab import files
import os

print("Click Choose Files below and select your kaggle.json")
uploaded = files.upload()

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle API key installed!")


In [ ]:
import glob

os.makedirs("data", exist_ok=True)
print("Downloading RWF-2000 from Kaggle. Please wait about 3-5 minutes...")
!kaggle datasets download -d vbookshelf/rwf2000-video-database-for-violence-detection -p data/ --unzip

dataset_root = None
for root, dirs, _ in os.walk("data"):
    if "train" in dirs and "val" in dirs:
        if os.path.isdir(os.path.join(root, "train", "Fight")):
            dataset_root = root
            break

if dataset_root is None:
    for candidate in ["data/RWF-2000", "data/rwf-2000"]:
        if os.path.exists(candidate):
            dataset_root = candidate
            break

if dataset_root:
    print("Dataset found at: " + dataset_root)
    for split in ["train", "val"]:
        for cls in ["Fight", "NonFight"]:
            path = os.path.join(dataset_root, split, cls)
            if os.path.exists(path):
                count = len(glob.glob(path + "/*.avi")) + len(glob.glob(path + "/*.mp4"))
                print("  " + split + "/" + cls + ": " + str(count) + " videos")
else:
    print("Could not detect dataset root. Run: !find data -type d")


## 🏋️ Part 1: Train Pose-Based CNN+GRU Classifier (`temporal_v1.pt`)
This extracts 17 body pose keypoints from YOLOv8-pose and trains the temporal GRU sequence model.

In [ ]:
import glob
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Load YOLOv8-pose
pose_model = YOLO("yolov8n-pose.pt")
pose_model.to(device)

def extract_pose_features(video_path, num_frames=16):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return np.zeros((num_frames, 34), dtype=np.float32)
    
    indices = np.linspace(0, max(0, total_frames - 1), num_frames, dtype=int)
    frames = []
    current_idx = 0
    pos = 0
    
    while cap.isOpened() and pos < len(indices):
        ret, frame = cap.read()
        if not ret:
            break
        if current_idx == indices[pos]:
            frames.append(frame)
            pos += 1
        current_idx += 1
    cap.release()
    
    # Pad if missing frames
    while len(frames) < num_frames:
        frames.append(frames[-1] if frames else np.zeros((224, 224, 3), dtype=np.uint8))
        
    # Batch predict pose
    results = pose_model.predict(frames, verbose=False, device=device)
    window = []
    for res in results:
        feat = np.zeros(34, dtype=np.float32)
        if res.keypoints is not None and len(res.keypoints) > 0:
            kp = res.keypoints.xyn[0].cpu().numpy() # (17, 2)
            feat = kp.flatten()[:34]
        window.append(feat)
    return np.array(window, dtype=np.float32) # (16, 34)

print("Feature extractor ready!")

In [ ]:
# 2. Extract Features for All RWF-2000 Videos (Fast on GPU)
processed_dir = Path("data/processed")
for split in ["train", "val"]:
    for cls in ["normal", "violent"]:
        (processed_dir / split / cls).mkdir(parents=True, exist_ok=True)

def process_split(split_name):
    split_dir = Path(dataset_root) / split_name
    
    # Normal / NonFight (Class 0)
    nonfight_videos = glob.glob(str(split_dir / "NonFight" / "*.avi")) + glob.glob(str(split_dir / "NonFight" / "*.mp4"))
    for v in tqdm(nonfight_videos, desc=f"{split_name} NonFight"): 
        feat = extract_pose_features(v)
        name = Path(v).stem + ".npy"
        np.save(processed_dir / split_name / "normal" / name, feat)
        
    # Violent / Fight (Class 1)
    fight_videos = glob.glob(str(split_dir / "Fight" / "*.avi")) + glob.glob(str(split_dir / "Fight" / "*.mp4"))
    for v in tqdm(fight_videos, desc=f"{split_name} Fight"): 
        feat = extract_pose_features(v)
        name = Path(v).stem + ".npy"
        np.save(processed_dir / split_name / "violent" / name, feat)

print("Processing Training Set...")
process_split("train")
print("Processing Validation Set...")
process_split("val")
print("✅ Feature extraction complete!")

In [ ]:
# 3. Define TemporalClassifier Architecture (Exact match to SmartOmniSentinel)
class TemporalClassifier(nn.Module):
    def __init__(self, feature_dim=34, hidden_dim=64, num_layers=1, num_classes=2, dropout=0.5):
        super().__init__()
        self.frame_encoder = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 128),
            nn.ReLU(inplace=True),
        )
        self.gru = nn.GRU(
            input_size=128,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )
        
    def forward(self, x):
        batch, seq, feat = x.shape
        x_flat = x.view(batch * seq, feat)
        encoded = self.frame_encoder(x_flat)
        encoded = encoded.view(batch, seq, -1)
        gru_out, _ = self.gru(encoded)
        last_hidden = gru_out[:, -1, :]
        return self.classifier(last_hidden)

# Dataset Loader for .npy files
class NpyDataset(Dataset):
    def __init__(self, split_dir, augment=False):
        self.samples = []
        self.augment = augment
        for f in glob.glob(str(Path(split_dir) / "normal" / "*.npy")):
            self.samples.append((f, 0))
        for f in glob.glob(str(Path(split_dir) / "violent" / "*.npy")):
            self.samples.append((f, 1))
            
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        arr = np.load(path).astype(np.float32)
        if self.augment:
            # Horizontal flip
            if np.random.rand() < 0.5:
                arr[:, 0:34:2] = 1.0 - arr[:, 0:34:2]
            # Gaussian noise
            arr += np.random.normal(0, 0.01, arr.shape).astype(np.float32)
        return torch.tensor(arr), label

train_ds = NpyDataset("data/processed/train", augment=True)
val_ds = NpyDataset("data/processed/val", augment=False)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
print(f"Loaded {len(train_ds)} train samples and {len(val_ds)} val samples.")

In [ ]:
# 4. Train the Temporal Model (Takes ~30 seconds on GPU!)
os.makedirs("ml/checkpoints", exist_ok=True)
model = TemporalClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

best_val_acc = 0.0
checkpoint_path = "ml/checkpoints/temporal_v1.pt"

for epoch in range(40):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(y)
        preds = out.argmax(dim=-1)
        correct += (preds == y).sum().item()
        total += len(y)
    scheduler.step()
    
    train_acc = correct / total
    
    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            preds = out.argmax(dim=-1)
            val_correct += (preds == y).sum().item()
            val_total += len(y)
    val_acc = val_correct / val_total
    
    if (epoch + 1) % 5 == 0 or val_acc > best_val_acc:
        print(f"Epoch {epoch+1:02d}/40 | Train Loss: {total_loss/total:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
        
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_acc": val_acc,
        }, checkpoint_path)

print(f"\n🏆 Best Validation Accuracy: {best_val_acc*100:.2f}%")
print(f"💾 Saved best model to: {checkpoint_path}")

## 🎬 Part 2: Train 3D-CNN & Export ONNX (`violence_model.onnx`)
Fine-tunes ResNet3D-18 on raw video clips and exports to ONNX format.

In [ ]:
from torchvision.models.video import r3d_18, R3D_18_Weights

os.makedirs("models", exist_ok=True)

class RWFVideoDataset(Dataset):
    def __init__(self, root_dir, split="train", num_frames=16, resize=(112, 112)):
        self.root_dir = Path(root_dir) / split
        self.num_frames = num_frames
        self.resize = resize
        self.video_paths = []
        self.labels = []
        
        for f in glob.glob(str(self.root_dir / "Fight" / "*.avi")) + glob.glob(str(self.root_dir / "Fight" / "*.mp4")):
            self.video_paths.append(f)
            self.labels.append(1.0)
        for f in glob.glob(str(self.root_dir / "NonFight" / "*.avi")) + glob.glob(str(self.root_dir / "NonFight" / "*.mp4")):
            self.video_paths.append(f)
            self.labels.append(0.0)
            
    def __len__(self):
        return len(self.video_paths)
        
    def __getitem__(self, idx):
        path = self.video_paths[idx]
        label = self.labels[idx]
        cap = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        indices = np.linspace(0, max(0, total - 1), self.num_frames, dtype=int)
        frames = []
        pos = 0
        cur = 0
        while cap.isOpened() and pos < len(indices):
            ret, frame = cap.read()
            if not ret: break
            if cur == indices[pos]:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, self.resize)
                frames.append(frame.astype(np.float32) / 255.0)
                pos += 1
            cur += 1
        cap.release()
        while len(frames) < self.num_frames:
            frames.append(frames[-1] if frames else np.zeros((112, 112, 3), dtype=np.float32))
            
        # Shape: (Channels=3, Time=16, Height=112, Width=112)
        tensor = torch.tensor(np.array(frames), dtype=torch.float32).permute(3, 0, 1, 2)
        return tensor, torch.tensor(label, dtype=torch.float32)

cnn_train_ds = RWFVideoDataset(dataset_root, split="train")
cnn_val_ds = RWFVideoDataset(dataset_root, split="val")
cnn_train_loader = DataLoader(cnn_train_ds, batch_size=8, shuffle=True, num_workers=2)
cnn_val_loader = DataLoader(cnn_val_ds, batch_size=8, shuffle=False, num_workers=2)

# Setup R3D-18
r3d_model = r3d_18(weights=R3D_18_Weights.DEFAULT)
r3d_model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(r3d_model.fc.in_features, 1))
r3d_model = r3d_model.to(device)

bce_criterion = nn.BCEWithLogitsLoss()
cnn_optimizer = torch.optim.Adam(r3d_model.parameters(), lr=1e-4, weight_decay=1e-3)
scaler = torch.cuda.amp.GradScaler()

print("Training R3D-18 for 6 epochs on GPU (Fast Mixed Precision)...\n")
for ep in range(6):
    r3d_model.train()
    for inp, lbl in tqdm(cnn_train_loader, desc=f"Epoch {ep+1}/6"):
        inp, lbl = inp.to(device), lbl.to(device)
        cnn_optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            out = r3d_model(inp).squeeze(1)
            loss = bce_criterion(out, lbl)
        scaler.scale(loss).backward()
        scaler.step(cnn_optimizer)
        scaler.update()

# Export to ONNX
print("\nExporting to models/violence_model.onnx...")
class ONNXWrapper(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return torch.sigmoid(self.m(x))

onnx_model = ONNXWrapper(r3d_model).eval()
dummy = torch.randn(1, 3, 16, 112, 112, device=device)
onnx_out_path = "models/violence_model.onnx"
torch.onnx.export(
    onnx_model, dummy, onnx_out_path,
    export_params=True, opset_version=11, do_constant_folding=True,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
print(f"✅ Successfully exported {onnx_out_path}!")

## 📥 Part 3: Download Trained Models to Your Laptop
Run this final cell to download both model files straight to your browser downloads folder.

In [ ]:
from google.colab import files

print("Initiating download for temporal_v1.pt...")
files.download("ml/checkpoints/temporal_v1.pt")

print("Initiating download for violence_model.onnx...")
files.download("models/violence_model.onnx")

print("\n🎉 Training and download complete!")
print("Place temporal_v1.pt into: smart-omnisentinel/ml/checkpoints/")
print("Place violence_model.onnx into: smart-omnisentinel/models/")